# Benchmark RAG — Top 10 por PDF e média/mediana por configuração

Lê os cinco arquivos `benchmark_rag_multidocumentos_*.csv` da pasta `analise_benchmark/voyage`. Não faz chamadas à API, não altera CSVs e não analisa chunks.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Execute o notebook na pasta "rag" do projeto, ou ajuste PASTA_RESULTADOS.
PASTA_RESULTADOS = Path.cwd() / "analise_benchmark" / "voyage"
ARQUIVOS = sorted(PASTA_RESULTADOS.glob("benchmark_rag_multidocumentos_*.csv"))
if not ARQUIVOS:
    raise FileNotFoundError(f"Nenhum CSV encontrado em {PASTA_RESULTADOS.resolve()}")

quadros = []
for arquivo in ARQUIVOS:
    tabela = pd.read_csv(arquivo, encoding="utf-8-sig")
    tabela.insert(0, "PDF", arquivo.stem.removeprefix("benchmark_rag_multidocumentos_"))
    quadros.append(tabela)
dados = pd.concat(quadros, ignore_index=True)
print(f"PDFs encontrados: {len(ARQUIVOS)} | Configurações registradas: {len(dados)}")
for arquivo in ARQUIVOS:
    print(" •", arquivo.name)


In [ ]:
CHAVE = ["Chunk Size (chars)", "Overlap (chars)", "K Retriever", "Top N Reranker"]
for coluna in CHAVE:
    if coluna not in dados:
        raise KeyError(f"Coluna necessária ausente: {coluna}")

# Priorizar relevância de páginas apenas quando houver anotações válidas
metricas_pagina = ["nDCG@N Páginas", "Recall@N Páginas", "Precision@N Páginas"]
if "Perguntas com Páginas Anotadas" in dados:
    tem_anotacao = pd.to_numeric(dados["Perguntas com Páginas Anotadas"], errors="coerce").fillna(0) > 0
else:
    tem_anotacao = pd.Series(False, index=dados.index)

if tem_anotacao.all() and all(c in dados.columns for c in metricas_pagina):
    CRITERIOS = metricas_pagina
    print("Critério do ranking: nDCG@N, Recall@N e Precision@N (páginas anotadas).")
else:
    CRITERIOS = [c for c in ["Taxa Acerto Documento Esperado", "MRR Documento Esperado"] if c in dados and pd.to_numeric(dados[c], errors="coerce").notna().all()]
    if not CRITERIOS:
        raise ValueError("Não há métricas de relevância completas para classificar todos os PDFs. Confira as colunas de avaliação.")
    print("Critério do ranking:", " > ".join(CRITERIOS))
    if not tem_anotacao.all():
        print("Aviso: métricas de páginas não estão anotadas em todas as linhas; não foram usadas para ordenar.")

LATENCIA = "Tempo Recuperação + Reranking Médio (s)"
ORDENAR = CRITERIOS + ([LATENCIA] if LATENCIA in dados else [])
ASC = [False] * len(CRITERIOS) + ([True] if LATENCIA in dados else [])

# Comparação justa: métricas ausentes não são tratadas como zero.
for coluna in ORDENAR:
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")
if dados[CRITERIOS].isna().any().any():
    raise ValueError("Há valores ausentes nas métricas usadas para classificar. Verifique os CSVs antes de comparar.")

ranking = dados.sort_values(ORDENAR, ascending=ASC, kind="stable").copy()
ranking["Posição no PDF"] = ranking.groupby("PDF").cumcount() + 1
top10 = ranking.loc[ranking["Posição no PDF"] <= 10].copy()

colunas_top = ["PDF", "Posição no PDF"] + CHAVE + [c for c in ["Taxa Acerto Documento Esperado", "MRR Documento Esperado", *metricas_pagina, LATENCIA, "Contexto Médio (chars)"] if c in top10.columns]
print("TOP 10 DE CADA PDF")
display(top10[colunas_top].sort_values(["PDF", "Posição no PDF"]).reset_index(drop=True))


In [ ]:
# Médias e medianas por configuração, considerando TODAS as observações
# disponíveis dos PDFs (inclusive quando uma configuração ficou fora do Top 10
# em algum documento). Assim não enviesamos as médias escolhendo só vitórias.
metricas = [c for c in ["Taxa Acerto Documento Esperado", "MRR Documento Esperado", *metricas_pagina, LATENCIA, "Contexto Médio (chars)", "Tempo Indexação (s)", "Score Reranker Médio"] if c in dados.columns]
for c in metricas:
    dados[c] = pd.to_numeric(dados[c], errors="coerce")

chaves_top = top10[CHAVE].drop_duplicates()
selecionadas = dados.merge(chaves_top, on=CHAVE, how="inner")
agregacoes = {"PDF": "nunique"}
agregacoes.update({m: ["mean", "median"] for m in metricas})
resumo = selecionadas.groupby(CHAVE, dropna=False).agg(agregacoes)
resumo.columns = ["PDFs avaliados" if c[0] == "PDF" else f"{c[0]} — {'Média' if c[1] == 'mean' else 'Mediana'}" for c in resumo.columns]
resumo = resumo.reset_index()

# Prioriza a mesma métrica do ranking individual na comparação agregada.
ordem_geral = [f"{m} — Média" for m in CRITERIOS]
resumo = resumo.sort_values(ordem_geral, ascending=[False] * len(ordem_geral)).reset_index(drop=True)
print(f"MÉDIA E MEDIANA — {len(resumo)} configurações distintas que apareceram em algum Top 10")
print("As estatísticas de cada configuração usam todos os PDFs em que ela foi avaliada.")
display(resumo.round(4))
